# Multiple Linear Regression Using Gradient Descent — Mathematical Implementation

This notebook implements **batch gradient descent from scratch** for multiple input features (`x1, x2, ..., xn`) and one output `y`.

The model is still:

`ŷ = w1*x1 + w2*x2 + ... + wn*xn + b`

or in vectorized form:

`ŷ = Xw + b`

Unlike the Normal Equation notebook, here the parameters `w` and `b` are learned iteratively by taking small steps in the direction that reduces the cost — the same idea as the single-variable gradient descent notebook, now generalized to a weight *vector* instead of a single weight.

## 1. Imports and dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
df = pd.DataFrame({
    "Hours_Studied":          [2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "Sleep_Hours":            [6, 7, 6, 8, 7, 6, 8, 7, 8, 6],
    "Attendance_Percentage":  [60, 65, 70, 72, 78, 80, 85, 88, 90, 95],
    "Marks":                  [42, 48, 51, 58, 63, 66, 74, 78, 85, 88]
})

df

In [ ]:
x_raw = df[["Hours_Studied", "Sleep_Hours", "Attendance_Percentage"]].values
y = df["Marks"].values

print("x_raw shape:", x_raw.shape)
print("y shape:", y.shape)


## 2. Feature scaling

With a single feature, gradient descent works fine on raw values. With multiple features on very different scales (e.g. `Hours_Studied` ranges from 2–11, while `Attendance_Percentage` ranges from 60–95), gradient descent can converge very slowly or become unstable.

We apply **z-score normalization** to each feature:

`x_scaled = (x - mean(x)) / std(x)`

This gives every feature a mean of 0 and a standard deviation of 1, so gradient descent takes similarly sized, well-behaved steps across all features.

In [ ]:
x_mean = x_raw.mean(axis=0)
x_std = x_raw.std(axis=0)

x = (x_raw - x_mean) / x_std

print("Feature means:", x_mean)
print("Feature stds :", x_std)
print("\nScaled features (first 3 rows):")
print(x[:3])


## 3. Multiple linear regression model

For `n` input features, computed for the whole dataset at once:

`ŷ = Xw + b`

where `X` is the scaled feature matrix and `w` is a vector with one weight per feature.

In [ ]:
def predict(X, w, b):
    return X @ w + b


## 4. Cost function

The same squared-error cost as the single-variable case, now computed over a weight vector:

`J(w,b) = (1 / (2m)) Σ(ŷᵢ - yᵢ)²`

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    predictions = predict(X, w, b)
    return np.sum((predictions - y) ** 2) / (2 * m)


## 5. Gradients

The partial derivatives of the cost function generalize directly from the single-variable case.

For each weight `wj`:

`∂J/∂wj = (1/m) Σ(ŷᵢ - yᵢ) * xᵢⱼ`

For the bias `b`:

`∂J/∂b = (1/m) Σ(ŷᵢ - yᵢ)`

Written in vectorized form for all weights at once:

`∂J/∂w = (1/m) Xᵀ(Xw + b - y)`

`∂J/∂b = (1/m) Σ(Xw + b - y)`

In [ ]:
def compute_gradients(X, y, w, b):
    m = X.shape[0]
    predictions = predict(X, w, b)
    error = predictions - y

    dw = (X.T @ error) / m
    db = np.sum(error) / m

    return dw, db


## 6. Gradient descent update rule

Each weight and the bias are updated simultaneously using the learning rate `α`:

`w := w - α(∂J/∂w)`

`b := b - α(∂J/∂b)`

This is the exact same update rule as the single-variable notebook — the only difference is that `w` is now a vector, so every weight is updated at once in a single vectorized operation.

In [ ]:
def gradient_descent(X, y, w_init, b_init, alpha, iterations):
    w = w_init.copy()
    b = b_init
    cost_history = []

    for i in range(iterations):
        dw, db = compute_gradients(X, y, w, b)

        w = w - alpha * dw
        b = b - alpha * db

        cost = compute_cost(X, y, w, b)
        cost_history.append(cost)

    return w, b, cost_history


## 7. Run gradient descent

We initialize `w` and `b` to zero and run a fixed number of iterations with a chosen learning rate `α`.

In [ ]:
w_init = np.zeros(x.shape[1])
b_init = 0.0

alpha = 0.1
iterations = 1000

gd_w, gd_b, cost_history = gradient_descent(x, y, w_init, b_init, alpha, iterations)

print("Learned w (scaled features):", gd_w)
print("Learned b:", gd_b)
print(f"Final cost: {cost_history[-1]:.6f}")


## 8. Visualize the cost over iterations

A correctly implemented gradient descent should show the cost steadily decreasing and flattening out as it converges.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(iterations), cost_history)
plt.xlabel("Iteration")
plt.ylabel("Cost J(w, b)")
plt.title("Cost vs Iteration — Gradient Descent")
plt.grid(True)
plt.show()


## 9. Predictions

`predict()` uses the same scaled features that gradient descent was trained on.

In [ ]:
gd_predictions = predict(x, gd_w, gd_b)

df["Predicted"] = gd_predictions

df

## 10. Visualize predicted vs actual values

As with the Normal Equation notebook, we compare predicted and actual values against the diagonal `ŷ = y` line, since a single 2D regression line is not possible with multiple features.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, gd_predictions, label="Predictions")

min_val = min(y.min(), gd_predictions.min()) - 2
max_val = max(y.max(), gd_predictions.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", label="Perfect Prediction (y = ŷ)")

plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("Predicted vs Actual — Gradient Descent")
plt.legend()
plt.grid(True)
plt.show()


## 11. Residual visualization

In [ ]:
residuals = y - gd_predictions

plt.figure(figsize=(8, 5))
plt.bar(range(len(residuals)), residuals)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Training example index")
plt.ylabel("Residual (y - ŷ)")
plt.title("Residuals — Gradient Descent")
plt.grid(True)
plt.show()


## 12. Mean Squared Error (MSE)

`MSE = (1/m) Σ(yᵢ - ŷᵢ)²`

In [ ]:
def compute_mse(y, predictions):
    return np.mean((y - predictions) ** 2)

mse = compute_mse(y, gd_predictions)

print(f"MSE : {mse:.6f}")
print(f"Cost: {compute_cost(x, y, gd_w, gd_b):.6f}")
print(f"MSE / 2: {mse / 2:.6f}")


## 13. Root Mean Squared Error (RMSE)

`RMSE = √MSE`

In [ ]:
def compute_rmse(y, predictions):
    return np.sqrt(compute_mse(y, predictions))

rmse = compute_rmse(y, gd_predictions)

print(f"RMSE: {rmse:.6f}")


## 14. Mean Absolute Error (MAE)

`MAE = (1/m) Σ|yᵢ - ŷᵢ|`

In [ ]:
def compute_mae(y, predictions):
    return np.mean(np.abs(y - predictions))

mae = compute_mae(y, gd_predictions)

print(f"MAE: {mae:.6f}")


## 15. R² score

`R² = 1 - [Σ(yᵢ - ŷᵢ)² / Σ(yᵢ - ȳ)²]`

In [ ]:
def compute_r2(y, predictions):
    ss_res = np.sum((y - predictions) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - (ss_res / ss_tot)

r2 = compute_r2(y, gd_predictions)

print(f"R²: {r2:.6f}")


## 16. Predict a new value

Because gradient descent was trained on **scaled** features, any new input must be scaled the same way — using the same `x_mean` and `x_std` computed from the training data — before being passed to `predict()`.

In [ ]:
new_x_raw = np.array([9, 7, 82])  # Hours_Studied, Sleep_Hours, Attendance_Percentage
new_x_scaled = (new_x_raw - x_mean) / x_std

new_prediction = predict(new_x_scaled, gd_w, gd_b)

print("Input (raw):", new_x_raw)
print(f"Predicted Marks: {new_prediction:.4f}")


## 17. Comparing gradient descent with the Normal Equation

Gradient descent is an **iterative, approximate** approach — it converges toward the optimum over many steps and depends on the learning rate and number of iterations.

The Normal Equation is a **direct, exact** approach — it solves for the optimum in one step using linear algebra.

For small datasets like this one, both methods should arrive at very similar costs. Gradient descent becomes more practical than the Normal Equation as the number of features grows very large, since matrix inversion becomes computationally expensive.

## 18. Final evaluation summary

In [ ]:
print("w (weights, scaled features):", gd_w)
print(f"b (bias)                     : {gd_b:.6f}")
print(f"Final cost                   : {cost_history[-1]:.6f}")
print(f"MSE                          : {mse:.6f}")
print(f"RMSE                         : {rmse:.6f}")
print(f"MAE                          : {mae:.6f}")
print(f"R²                           : {r2:.6f}")
